# torch nn.Embedding

In [1]:
import nltk

nltk.download('punkt')  # NLTK 토크나이저
nltk.download('punkt_tab') # punkt 관련 테이블리소스
nltk.download('stopwords') # 불용어 목록

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\MoonSungHo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\MoonSungHo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\MoonSungHo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

# 사전학습된 임베딩 사용하지 않은 경우

In [2]:
sentences = [          
    'nice great best amazing',  # 긍정 문장 예시
    'stop lies',                # 부정/비판 문장 예시
    'pitiful nerd',             # 부정 문장 예시
    'excellent work',           # 긍정 문장 예시
    'supreme quality',          # 긍정 문장 예시
    'bad',                      # 부정 문장 예시
    'highly respectable'        # 긍정 문장 예시
]                               # 분류 모델에 넣을 입력 문장 리스트(list[str])
labels = [1, 0, 0, 1, 1, 0, 1]  # 각 문장에 대한 이진 라벨(1=긍정, 0=부정)

In [3]:
# NLTK 토그나이저로 토큰화
from nltk.tokenize import word_tokenize

tokenized_sentences = [word_tokenize(sent) for sent in sentences]
tokenized_sentences

[['nice', 'great', 'best', 'amazing'],
 ['stop', 'lies'],
 ['pitiful', 'nerd'],
 ['excellent', 'work'],
 ['supreme', 'quality'],
 ['bad'],
 ['highly', 'respectable']]

In [4]:
# 단어 사전 생성 + 정수 인코딩
from collections import Counter

tokens = [token for sent in tokenized_sentences for token in sent]  # 문장 리스트를 1차원으로 평탄화
word_counts = Counter(tokens)   # 전체 토큰의 등장 갯수
print(word_counts)

word_to_index = {word : index + 2 for index, word in enumerate(tokens)} # 토큰을 순서대로 인덱싱 (인덱스 +2)
word_to_index['<PAD>'] = 0  # 패딩 토큰 추가
word_to_index['<UNK>'] = 1  # OOV 토큰 추가
word_to_index = dict(sorted(word_to_index.items(), key=lambda x:x[1])) # 딕셔너리 정렬(인덱스 순)
print(word_to_index)

vocab_size = len(word_to_index) # 특수토큰 포함 전체 어휘 수
vocab_size

Counter({'nice': 1, 'great': 1, 'best': 1, 'amazing': 1, 'stop': 1, 'lies': 1, 'pitiful': 1, 'nerd': 1, 'excellent': 1, 'work': 1, 'supreme': 1, 'quality': 1, 'bad': 1, 'highly': 1, 'respectable': 1})
{'<PAD>': 0, '<UNK>': 1, 'nice': 2, 'great': 3, 'best': 4, 'amazing': 5, 'stop': 6, 'lies': 7, 'pitiful': 8, 'nerd': 9, 'excellent': 10, 'work': 11, 'supreme': 12, 'quality': 13, 'bad': 14, 'highly': 15, 'respectable': 16}


17

In [7]:
# 토큰화된 문장 리스트를 받아 단어 -> 인덱스 사전
def texts_to_sequences(sentences, word_to_index):
    sequences = []
    
    for sent in sentences:          # 문장 단위 순회
        sequence = []
        
        for token in sent:          # 토큰 단위 순회
            if token in word_to_index:      # 사전에 있는 단어면 
                sequence.append(word_to_index[token])   # 해당 단어의 값(ID) 추가
            else:                           # 사전에 없으면
                sequence.append(word_to_index['<UNK>']) # 해당 위치에 OOV 토큰 추가
            
        sequences.append(sequence)
    
    return sequences

sequences = texts_to_sequences(tokenized_sentences, word_to_index)
sequences

[[2, 3, 4, 5], [6, 7], [8, 9], [10, 11], [12, 13], [14], [15, 16]]

In [11]:
# 패딩 추가
import numpy as np

# 서로 다른 길이의 정수 시퀀스를 0(<PAD>)으로 채워 (문장수, maxlen) 형태로 맞추는 함수
def pad_sequences(sequences, maxlen):
    padded_sequences = np.zeros((len(sequences), maxlen), dtype=int)    # (문장수 x maxlen) 크기의 0패딩 생성
    
    for idx, seq in enumerate(sequences):
        # idx 번째 행에서 0번 위치부터 len(seq) -1 위치 까지를 seq의 처음부터 maxlen개 까지 사용
        padded_sequences[idx, : len(seq)] = seq[ :maxlen]   # 앞에서부터 시퀀스 채움. 시퀀스가 길면 maxlen으로 자름
    
    return padded_sequences

padded_sequences = pad_sequences(sequences, maxlen= 4)

padded_sequences    # (문장수, maxlen=4) 형태의 정수 배열


array([[ 2,  3,  4,  5],
       [ 6,  7,  0,  0],
       [ 8,  9,  0,  0],
       [10, 11,  0,  0],
       [12, 13,  0,  0],
       [14,  0,  0,  0],
       [15, 16,  0,  0]])

In [13]:
padded_sequences.shape      # 문장수, 고정길이 4

(7, 4)

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# 정수 시퀀스를 임베딩 -> RNN -> 선형층으로 처리해서 이진분류 logit(1개) 출력하는 모델
class SimpleNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_size):
        super().__init__()
        
        # 단어 ID를 밀집 벡터로 변환하는 임베딩
        self.embedding = nn.Embedding(
            num_embeddings= vocab_size,     # 단어 사전 크기
            embedding_dim= embedding_dim,   # 임베딩 차원
            padding_idx= 0                  # 패딩 0 인덱스는 업데이트 하지 않음
        )
        
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first= True)    # RNN 입력(배치, 길이, 차원)
        # 마지막 은닉 상태를 받아 1차원 logit값으로 변환(차후 BCEWithLogitsLoss) 등 사용해서 확률값 변환해야
        self.out = nn.Linear(hidden_size, 1)        
        
    def forward(self, x):
        embedded = self. embedding(x)   # (batch, seq_len) -> (batch, seq_len, embedding_dim) 임베딩차원이 늘어남
        out, h_n = self.rnn(embedded)   # out: logit 값, h_n : (num_layers*directions, batch, hidden_size)
        
        out = self.out(h_n.squeeze(0))  # 첫번째 차원 num_layer를 제거하는 역할
        return out

embedding_dim = 100
model = SimpleNet(vocab_size, embedding_dim, hidden_size= 16)
model

SimpleNet(
  (embedding): Embedding(17, 100, padding_idx=0)
  (rnn): RNN(100, 16, batch_first=True)
  (out): Linear(in_features=16, out_features=1, bias=True)
)

In [17]:
from torchinfo import summary

summary(model)  # 모델의 레이어 구성/ 파라미터 수 요약 정보

Layer (type:depth-idx)                   Param #
SimpleNet                                --
├─Embedding: 1-1                         1,700
├─RNN: 1-2                               1,888
├─Linear: 1-3                            17
Total params: 3,605
Trainable params: 3,605
Non-trainable params: 0

In [ ]:
# 임베딩 가중치 확인
import pandas as pd

wv = model.embedding.weight.data    # 임베딩 층의 가중치 행렬 (단어ID x 임베딩차원)
print(wv.shape)     # (vocab_size, embedding_dim)

vocab = word_to_index.keys()    # 인덱스만 가져옴
pd.DataFrame(wv, index= vocab)  # 인덱스 추가해서 데이터프레임으로 확인

torch.Size([17, 100])


,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
<PAD>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
<UNK>,-1.489151,-0.478168,0.624665,0.841868,0.302996,-0.196575,0.074802,-0.014765,1.993011,0.248462,...,-0.290707,0.267458,1.863307,-0.463269,-0.612702,1.873655,1.010165,-1.279777,-0.604928,-0.036765
nice,-1.940840,-0.642235,-0.827230,-0.153480,0.601862,-0.961650,-0.572814,-1.130835,-1.741404,0.462255,...,-0.098509,0.701785,-0.122675,0.256015,-0.364284,-0.959921,-1.055997,1.510167,0.881846,1.254829
great,0.266571,1.109121,-0.103469,0.958548,-1.420372,-0.102787,-0.978213,-1.468055,-0.176750,-0.282300,...,-0.028804,1.645970,0.247093,-1.673830,-1.974807,1.009579,-0.077543,-0.173353,0.199645,0.033100
best,-0.933488,0.036423,0.389137,-1.199614,-0.545652,0.576851,0.712901,-0.073388,1.892638,0.051037,...,-1.671050,0.397759,-0.269974,-0.682993,-0.980508,0.314316,-0.075183,0.455393,-0.041866,0.800663
amazing,-0.802324,1.366081,-0.766604,-0.363428,0.147061,-0.625199,-0.197637,1.056249,0.906132,0.641239,...,0.037857,-1.133987,-1.064417,-2.004746,0.206663,0.786997,1.655770,-0.083211,-1.324980,-1.247996
stop,-0.300566,0.268717,1.249191,-1.025134,-2.002704,-1.636861,2.699641,-0.631789,0.257419,-0.644794,...,-0.180987,2.082733,0.018053,-0.661450,-0.867946,-0.541498,-1.639357,0.620544,-1.625124,0.668543
lies,-0.446992,-0.563913,-1.109903,1.973458,-1.169474,0.397055,0.010495,-1.994676,0.293116,1.418192,...,-1.160437,-1.722566,-1.165543,0.663240,0.043851,0.165986,-0.497862,1.302298,-0.826841,0.433975
pitiful,-1.040376,-0.325287,0.575160,-0.845914,0.882306,-0.049669,1.713145,1.531182,0.647034,2.237503,...,1.399472,-2.858528,-0.646437,0.006129,-0.779491,0.997822,0.492645,-1.828195,-0.143877,0.164490
nerd,-0.768273,-0.411789,-0.243556,0.859529,-0.027865,1.270614,1.100614,-0.831581,-0.686426,0.495083,...,0.787037,0.909169,0.732238,0.904475,-0.256120,-0.804150,1.162628,-0.120081,0.262176,-0.373045


In [19]:
X = torch.tensor(padded_sequences, dtype= torch.long)
y = torch.tensor(labels, dtype=torch.float).unsqueeze(1)

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=2, shuffle= True)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(),lr=0.005)

In [23]:
for epoch in range(20):
    epoch_loss = 0
    
    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()           # 이전 기울기 초기화
        output = model(x_batch)         # 순전파
        loss = criterion(output, y_batch)        # 손실 계산
        loss.backward()                 # 역전파 : 기울기 계산
        optimizer.step()                # 파라미터 업데이트
        
        epoch_loss += loss.item()   # 미니배치 손실은 python float 형태로 누적
        
    print(f'Epoch {epoch+1} Loss : {epoch_loss/ len(dataloader)}')  # epoch 손실

Epoch 1 Loss : 0.7268701046705246
Epoch 2 Loss : 0.5872670114040375
Epoch 3 Loss : 0.46925439685583115
Epoch 4 Loss : 0.3838958963751793
Epoch 5 Loss : 0.29036322236061096
Epoch 6 Loss : 0.21101030334830284
Epoch 7 Loss : 0.15631529316306114
Epoch 8 Loss : 0.11123323626816273
Epoch 9 Loss : 0.08045494370162487
Epoch 10 Loss : 0.05968546308577061
Epoch 11 Loss : 0.04415831947699189
Epoch 12 Loss : 0.035805572755634785
Epoch 13 Loss : 0.030565641820430756
Epoch 14 Loss : 0.025872881524264812
Epoch 15 Loss : 0.022780620492994785
Epoch 16 Loss : 0.01911601470783353
Epoch 17 Loss : 0.017549747601151466
Epoch 18 Loss : 0.015392675064504147
Epoch 19 Loss : 0.014027644414454699
Epoch 20 Loss : 0.012796663213521242


In [ ]:
# 평가 및 예측
model.eval()

with torch.no_grad():       # 기울기 계산 비활성화
    output = model(X)       # 
    prob = torch.sigmoid(output)    # 0~1 사이 
    pred = (prob >= 0.5).int()      # 임계값 0.5 가준으로 이진분류(0/1)
    
print(labels)
print(pred.squeeze().detach().numpy())

[1, 0, 0, 1, 1, 0, 1]
[1 0 0 1 1 0 1]


# 사전학습된 임베딩 모델을 사용

In [30]:
from gensim.models import KeyedVectors

# 사전 학습된 Word2Vec 모델 로드
model_wv = KeyedVectors.load_word2vec_format(
    'GoogleNews-vectors-negative300.bin',    # 뉴스 파일 로드
    binary=True     # 바이너리 파일형식
)
model_wv.vectors.shape

(3000000, 300)

In [32]:
# 임베딩 메트릭스 초기화 후 사전학습 임베딩 차원으로 재구성
print(len(word_to_index))

# (vocab_size, embedding_dim) 크기의 0행렬 생성
embedding_matrix = np.zeros((len(word_to_index), model_wv.vectors.shape[1]))    
embedding_matrix.shape

17


(17, 300)

In [35]:
def get_word_embedding(word):
    if word in model_wv:
        return model_wv[word]
    else:
        return None

get_word_embedding('nerd').shape

(300,)

In [36]:
for word, index in word_to_index.items():
    if index >= 2:
        emb = get_word_embedding(word)
        if emb is not None:
            embedding_matrix[index] = emb   #해당 단어 인덱스 위치에 사전학습 벡터를 복사

In [37]:
# 임베딩 메트릭스 확인
pd.DataFrame(embedding_matrix, index= word_to_index.keys())

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
<PAD>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
<UNK>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
nice,0.158203,0.105957,-0.189453,0.386719,0.083496,-0.267578,0.083496,0.113281,-0.104004,0.178711,...,-0.085449,0.189453,-0.146484,0.134766,-0.040771,0.032715,0.089355,-0.267578,0.008362,-0.213867
great,0.071777,0.208008,-0.028442,0.178711,0.132812,-0.099609,0.096191,-0.116699,-0.008545,0.148438,...,-0.011475,0.064453,-0.289062,-0.048096,-0.199219,-0.071289,0.064453,-0.167969,-0.020874,-0.142578
best,-0.126953,0.021973,0.287109,0.153320,0.127930,0.032715,-0.115723,-0.029541,0.153320,0.011292,...,0.006439,-0.033936,-0.166016,-0.016846,-0.048584,-0.022827,-0.152344,-0.101562,-0.090332,0.088379
amazing,0.073730,0.004059,-0.135742,0.022095,0.180664,-0.046631,0.224609,-0.229492,-0.040039,0.225586,...,0.018433,-0.021240,-0.250000,-0.020142,-0.310547,-0.207031,-0.006317,-0.141602,-0.150391,-0.137695
stop,-0.057861,0.013184,0.115234,0.069824,-0.306641,-0.044678,0.048584,0.152344,0.073242,-0.100098,...,0.100098,0.171875,-0.113281,0.064453,-0.115723,0.048096,-0.004822,0.086426,0.029907,0.007812
lies,0.149414,-0.012817,0.328125,0.025513,0.017334,0.190430,0.188477,-0.143555,-0.090820,0.206055,...,-0.308594,0.183594,-0.202148,0.031494,-0.164062,-0.201172,0.080078,-0.105469,0.149414,0.157227
pitiful,0.269531,0.253906,-0.020996,0.060303,-0.010925,0.217773,0.139648,-0.057617,0.312500,0.253906,...,-0.063477,0.132812,-0.094238,0.089355,-0.065430,-0.016235,-0.107910,-0.072266,-0.094238,0.028809
nerd,0.265625,-0.207031,-0.026611,0.419922,-0.208984,0.390625,0.164062,0.063965,0.149414,-0.017700,...,0.215820,0.125000,-0.227539,-0.310547,-0.112793,-0.096680,0.255859,0.124023,-0.030273,0.082031


In [44]:
## nn.Parameter 를 활용해 학습 가능 파라미터를 나눔(이론)

a = torch.tensor([1., 2., 3.], requires_grad=False)
b = torch.tensor([1., 2., 3.], requires_grad=True)

print(a.requires_grad)
print(b.requires_grad)

False
True


In [39]:
# 정수 시퀀스를 임베딩 -> RNN -> 선형층으로 처리해서 이진분류 logit(1개) 출력하는 모델
class SimpleNet(nn.Module):
    def __init__(self, vocab_size, embedding_dim,embedding_matrix , hidden_size):
        super().__init__()
        
        # 단어 ID를 밀집 벡터로 변환하는 임베딩
        self.embedding = nn.Embedding(
            num_embeddings= vocab_size,     # 단어 사전 크기
            embedding_dim= embedding_dim,   # 임베딩 차원
            padding_idx= 0                  # 패딩 0 인덱스는 업데이트 하지 않음
        )
        
        # 사전학습된 임베딩 벡터로 초기화
        self.embedding.weight = nn.Parameter(torch.tensor(embedding_matrix, dtype=torch.float))
        # self.embedding.weight.requires_grad = False     # True면 파인튜닝(추가학습), False 면 임베딩 고정 
        
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first= True)    # RNN 입력(배치, 길이, 차원)
        # 마지막 은닉 상태를 받아 1차원 logit값으로 변환(차후 BCEWithLogitsLoss) 등 사용해서 확률값 변환해야
        self.out = nn.Linear(hidden_size, 1)        
        
    def forward(self, x):
        embedded = self. embedding(x)   # (batch, seq_len) -> (batch, seq_len, embedding_dim) 임베딩차원이 늘어남
        out, h_n = self.rnn(embedded)   # out: logit 값, h_n : (num_layers*directions, batch, hidden_size)
        
        out = self.out(h_n.squeeze(0))  # 첫번째 차원 num_layer를 제거하는 역할
        return out

embedding_dim = model_wv.vectors.shape[1]
model = SimpleNet(vocab_size, embedding_dim, embedding_matrix, hidden_size= 16)
model

SimpleNet(
  (embedding): Embedding(17, 300, padding_idx=0)
  (rnn): RNN(300, 16, batch_first=True)
  (out): Linear(in_features=16, out_features=1, bias=True)
)

In [41]:
X = torch.tensor(padded_sequences, dtype= torch.long)
y = torch.tensor(labels, dtype=torch.float).unsqueeze(1)

dataset = TensorDataset(X, y)
dataloader = DataLoader(dataset, batch_size=2, shuffle= True)


criterion = nn.BCEWithLogitsLoss()          # 시그모이드를 포함한 손실함수
optimizer = optim.Adam(model.parameters(),lr=0.005)

In [42]:
for epoch in range(20):
    epoch_loss = 0
    
    for x_batch, y_batch in dataloader:
        optimizer.zero_grad()           # 이전 기울기 초기화
        output = model(x_batch)         # 순전파
        loss = criterion(output, y_batch)        # 손실 계산
        loss.backward()                 # 역전파 : 기울기 계산
        optimizer.step()                # 파라미터 업데이트
        
        epoch_loss += loss.item()   # 미니배치 손실은 python float 형태로 누적
        
    print(f'Epoch {epoch+1} Loss : {epoch_loss/ len(dataloader)}')  # epoch 손실

Epoch 1 Loss : 0.6967815160751343
Epoch 2 Loss : 0.5918821841478348
Epoch 3 Loss : 0.4988890588283539
Epoch 4 Loss : 0.3916431665420532
Epoch 5 Loss : 0.2922549694776535
Epoch 6 Loss : 0.19546743668615818
Epoch 7 Loss : 0.14188622683286667
Epoch 8 Loss : 0.1027082521468401
Epoch 9 Loss : 0.0762705858796835
Epoch 10 Loss : 0.056924959644675255
Epoch 11 Loss : 0.045010858215391636
Epoch 12 Loss : 0.03529566619545221
Epoch 13 Loss : 0.029379981569945812
Epoch 14 Loss : 0.024716605432331562
Epoch 15 Loss : 0.021309264469891787
Epoch 16 Loss : 0.01866410067304969
Epoch 17 Loss : 0.01658109575510025
Epoch 18 Loss : 0.014540239470079541
Epoch 19 Loss : 0.013481022557243705
Epoch 20 Loss : 0.012442019768059254


In [43]:
# 평가 및 예측
model.eval()

with torch.no_grad():       # 기울기 계산 비활성화
    output = model(X)       # 
    prob = torch.sigmoid(output)    # 0~1 사이 
    pred = (prob >= 0.5).int()      # 임계값 0.5 가준으로 이진분류(0/1)
    
print(labels)
print(pred.squeeze().detach().numpy())

[1, 0, 0, 1, 1, 0, 1]
[1 0 0 1 1 0 1]
